# XTTS-v2 fine-tune &mdash; VoiceMakers Sinhala female voices

Dinithi (4.82 h) + Harini (2.14 h) &asymp; **7 h across two distinct female speakers**.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** or **GPU P100** — either works; the scripts pin to one GPU |
| Internet | **ON** |
| Input | `SinhalaTTS_Dataset_Publication_by_VoiceMakers` (already attached) |
| Persistence | **Files** &mdash; needed to resume past the 12 h session limit |

## Two things this run does differently

**The text becomes ASCII before it reaches the tokenizer.** XTTS-v2's `vocab.json` is a
whitespace-pretokenised BPE with an `[UNK]` fallback and contains no Sinhala codepoint &mdash;
nor the diacritics this corpus romanises with (`ā ī ū ē ṭ ḍ ṇ ḷ ṁ` are all missing). Fed
either column raw, **every word becomes one `[UNK]`** and the model trains on "unknown
unknown unknown": loss falls, audio is noise. Cell 4 asserts 0 `[UNK]` before any GPU time
is spent.

**Two speakers, correctly labelled.** XTTS samples a conditioning clip from the *same
speaker* on every step. Two real labels teach "the reference predicts the voice"; pooling
them under one label teaches the opposite, and no amount of data repairs it.

## 1. Install &mdash; restart the session after this cell

In [ ]:
# coqui-tts is the maintained idiap fork. Do NOT `pip install TTS` -- that one
# pins torch<2.1 and replaces Kaggle's CUDA build with a CPU wheel.
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" soundfile librosa tensorboard

import os
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "\n*** NO GPU. Settings -> Accelerator -> GPU T4 x2 or P100, then restart. ***")

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch":
    # Save & Run All cannot restart the session, and does not need to: every step
    # that imports TTS runs in a fresh subprocess and so picks these packages up
    # regardless of what this notebook process already imported.
    print("\nBatch mode -- no restart needed, continuing straight on.")
else:
    print("\n>>> Now: Run -> Restart session, then continue from the NEXT cell. <<<")


## 2. Code and paths

In [ ]:
import os, shutil, subprocess, pathlib

REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
CODE = "/kaggle/working/Dataset-creation-withEmotion"

# Always clone fresh rather than pulling. With Persistence -> Files on, a stale
# checkout survives between sessions, and `git pull --ff-only` cannot fast-forward
# across a rewritten history -- it fails, and a swallowed failure would leave you
# running old code while believing it was current. A shallow clone costs seconds.
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
SRC = CODE + "/xtts_model_female"
print("code at", subprocess.run(["git", "-C", CODE, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

# Locate the attached dataset by finding the speaker folder and taking its
# parent. Kaggle nests inputs differently depending on how they were attached,
# so anchoring on a known folder name beats guessing the mount path.
inp = pathlib.Path("/kaggle/input")
hits = [p for p in inp.rglob("*") if p.is_dir() and "dinithi" in p.name.lower()] \
       if inp.is_dir() else []
DATA = str(min(hits, key=lambda p: len(p.parts)).parent) if hits else None

print("code   :", SRC)
print("data   :", DATA)
if DATA is None:
    raise RuntimeError(
        "Dataset not found. Attach 'SinhalaTTS Dataset Publication by VoiceMakers' "
        "via Add Input. Directories seen under /kaggle/input: "
        + str([p.name for p in inp.iterdir()] if inp.is_dir() else []))

# /kaggle/working is a 20 GB volume and a GPTTrainer checkpoint is ~5.5 GB --
# model plus AdamW state. The trainer holds up to four at once: best_model_<step>
# .pth, the full copy of it that trainer.io.save_best_model writes out as
# best_model.pth, one periodic checkpoint, and the next best model, written
# before the old one is deleted. That peaks at ~24 GB, so a run pointed at
# /kaggle/working dies of ENOSPC around global step 1760, roughly 45 min in --
# and it takes the notebook with it, because papermill writes
# __notebook__.ipynb to the same volume and a full disk truncates the JSON
# mid-write. Train on /kaggle/temp, which has hundreds of GB, and mirror back
# only the checkpoint that has to survive the session.
DATASET = "/kaggle/temp/female_dataset"     # scratch, not against the 20 GB quota
RUN     = "/kaggle/temp/run"                # checkpoints live here
KEEP    = "/kaggle/working/run"             # resume mirror; config + best model only
EVAL    = "/kaggle/working/eval_out"
BASE    = RUN + "/training/XTTS_v2.0_original_model_files"
os.makedirs("/kaggle/temp", exist_ok=True)


def sh(args, cwd=SRC):
    """Run a step and STOP the notebook if it fails.

    `!cmd` returns quietly on a non-zero exit, so a failed prepare step used to
    let the notebook march on and burn GPU minutes on training that could not
    possibly work. Under Save & Run All that is expensive and the real error
    scrolls far out of sight, so every step that others depend on goes through
    here instead.
    """
    print("$", " ".join(args), flush=True)
    r = subprocess.run(args, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f"step failed with exit code {r.returncode}: {' '.join(args)}")


## 3. Inspect the layout before trusting it

The published folders are inconsistent (`Isuru-44100Hz` vs `Yasindu-44100`, and at least
one speaker directory nested inside a duplicate of itself). Look at what is actually there.

In [ ]:
import pathlib, collections
root = pathlib.Path(DATA)
for d in sorted(p for p in root.rglob("*") if p.is_dir()):
    wavs = list(d.glob("*.wav"))
    csvs = list(d.glob("*.csv"))
    if wavs or csvs:
        print(f"{str(d.relative_to(root)):45s} {len(wavs):5d} wav  {[c.name for c in csvs]}")

meta = sorted(root.rglob("metadata.csv"))
print("\nmetadata files:", [str(m.relative_to(root)) for m in meta])
if meta:
    print("\nfirst 3 raw lines of", meta[0].name)
    for line in meta[0].read_text(encoding="utf-8-sig").splitlines()[:3]:
        print("  ", line[:160])

## 4. Build the dataset &mdash; and prove the text tokenises

This **fails loudly** rather than training on garbage if the romanisation contains a
character `sinhala_text.py` does not map, or if any `[UNK]` survives.

In [ ]:
import urllib.request
VOCAB = "/kaggle/temp/vocab.json"
urllib.request.urlretrieve(
    "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json", VOCAB)

sh(["python", "prepare_voicemakers.py",
    "--src", DATA, "--out", DATASET,
    "--speakers", "dinithi", "harini",
    "--vocab", VOCAB, "--eval-per-speaker", "40"])


## 5. Smoke test &mdash; two minutes, catches every wiring fault

In [ ]:
import subprocess, sys
sys.path.insert(0, SRC)
import train_log

# The smoke test exists to catch the one thing a zero exit code does NOT: a loss
# of nan. Training can run for hours writing NaN checkpoints and still exit 0.
# So parse the losses rather than trusting the return code -- see train_log.py,
# which strips the trainer's ANSI colour codes before doing so.
SMOKE_LOG = "/kaggle/working/smoke.log"
cmd = ["python", "train_xtts_female.py", "--dataset", DATASET,
       "--out", "/kaggle/temp/smoke", "--smoke",
       "--batch-size", "2", "--grad-accum", "2"]
print("$", " ".join(cmd), flush=True)
with open(SMOKE_LOG, "w") as fh:
    rc = subprocess.run(cmd, cwd=SRC, stdout=fh, stderr=subprocess.STDOUT).returncode

txt = open(SMOKE_LOG, encoding="utf-8", errors="replace").read()
print("\n".join(train_log.interesting(txt, 12)))

if rc != 0:
    print(train_log.strip_ansi(txt)[-3000:])
    raise RuntimeError(f"smoke test exited {rc} -- full output in {SMOKE_LOG}")

values = train_log.losses(txt)
if not values:
    raise RuntimeError("smoke test printed no loss_mel_ce -- training never started")
bad = train_log.nonfinite(values)
if bad:
    raise RuntimeError(
        f"loss_mel_ce is {bad[0]!r} -- this run would produce only NaN checkpoints.\n"
        "  If --mixed-precision was passed, drop it (it is off by default now).\n"
        "  If fp16 was already off, the cause is data: look for silent or\n"
        "  corrupt clips in the corpus.")
print(f"\nOK -- {len(values)} finite loss_mel_ce values, "
      f"first {values[0]} -> last {values[-1]}")


In [ ]:
# The smoke run writes the same ~5.5 GB checkpoints as the real one; drop them
# before training starts rather than carrying them for the next eight hours.
!rm -rf /kaggle/temp/smoke
!df -h /kaggle/working /kaggle/temp | grep -v Filesystem

## 6. Train

Backgrounded so the notebook stays responsive. Effective batch is
`batch_size x grad_accum = 64`. Upstream recommends 252, which is right for a datacentre;
on one T4 that is ~100 s per optimiser step and a whole session buys ~400 steps &mdash; far
too few to move the model onto a new sound inventory. Drop `--batch-size` to 3 or 2 on OOM
and raise `--grad-accum` to keep the product near 64.

In [ ]:
import os, shutil, subprocess, sys, time
sys.path.insert(0, SRC)
import train_log

# Free space on the volume holding `path`, in GB. Walks up to the nearest
# existing ancestor: RUN does not exist for the first seconds of a run, and
# disk_usage() on a missing path raises -- which would take out the very guard
# that is supposed to stop a crash.
def free_gb(path):
    path = os.path.abspath(path)
    while not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return shutil.disk_usage(path).free / 1e9

# Stop while there is still room to finish writing a checkpoint pair (~11 GB),
# and while /kaggle/working can still hold the notebook papermill writes at the
# end. Hitting ENOSPC instead loses the checkpoint being written AND the
# notebook, which is how the previous run lost its whole session.
RUN_FLOOR_GB, WORKING_FLOOR_GB = 12.0, 2.0

# Kaggle sets this to "Batch" under Save & Run All, "Interactive" otherwise.
BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive").lower() == "batch"
TRAIN_BUDGET_H = 8.5   # leave room for evaluation inside Kaggle's 12 h session cap

LOG = "/kaggle/working/train.log"
cmd = ["python", "train_xtts_female.py",
       "--dataset", DATASET, "--out", RUN,
       "--epochs", "40", "--batch-size", "4", "--grad-accum", "16",
       "--lr", "1e-5", "--save-step", "1000"]
print(" ".join(cmd))
print("mode:", "BATCH (blocking)" if BATCH else "INTERACTIVE (backgrounded)")

with open(LOG, "w") as fh:
    proc = subprocess.Popen(cmd, cwd=SRC, stdout=fh, stderr=subprocess.STDOUT)


def read_log():
    try:
        with open(LOG, encoding="utf-8", errors="replace") as fh:
            return fh.read()
    except OSError:
        return ""


if not BATCH:
    print("pid", proc.pid, "-- training in the background")
    print("Re-run the next cell to watch the loss.")
else:
    # Save & Run All: everything below needs a checkpoint, so this blocks. The
    # wall-clock cap matters because Kaggle kills the session at 12 h without
    # warning; checkpoints are written every save_step and best_model.pth at
    # every eval, so a capped run is still usable, just fewer epochs.
    deadline = time.time() + TRAIN_BUDGET_H * 3600
    last_beat, died_of_nan, out_of_disk = 0.0, False, None
    while proc.poll() is None and time.time() < deadline:
        time.sleep(30)
        # Checked every loop, not every heartbeat -- a checkpoint save can fill
        # the volume in well under the ten minutes between reports.
        if free_gb(RUN) < RUN_FLOOR_GB:
            out_of_disk = f"{RUN} is down to {free_gb(RUN):.1f} GB"
        elif free_gb("/kaggle/working") < WORKING_FLOOR_GB:
            out_of_disk = f"/kaggle/working is down to {free_gb('/kaggle/working'):.1f} GB"
        if out_of_disk:
            print(f"\ndisk guard: {out_of_disk} -- stopping now so the checkpoint "
                  "already on disk stays usable and the notebook can still be "
                  "written.", flush=True)
            break
        if time.time() - last_beat > 600:          # heartbeat every ~10 min
            last_beat = time.time()
            txt = read_log()
            left = (deadline - time.time()) / 3600
            print(f"[{left:5.2f} h left] "
                  f"[disk {free_gb(RUN):.0f} GB] "
                  + " | ".join(t[-160:] for t in train_log.interesting(txt, 2)),
                  flush=True)
            # Do not sit through eight hours of NaN. The smoke test should have
            # caught it, but belt and braces.
            if train_log.diverged(train_log.losses(txt)):
                died_of_nan = True
                print("\nloss_mel_ce has been nan for several reports -- aborting. "
                      "Every checkpoint from here would be NaN.", flush=True)
                break
    if proc.poll() is None:
        if not (died_of_nan or out_of_disk):
            print(f"\nBudget of {TRAIN_BUDGET_H} h reached -- stopping so evaluation "
                  "and export still get to run.", flush=True)
        proc.terminate()
        try:
            proc.wait(timeout=300)
        except subprocess.TimeoutExpired:
            proc.kill()
    print("\ntraining process exited with code", proc.returncode)

    # Mirror what a resume needs onto the persisted volume. --continue-path wants
    # a run directory holding config.json and a checkpoint; best_model.pth carries
    # the optimizer state too, so those two files are the whole resume kit. The
    # periodic checkpoints stay on /kaggle/temp and are discarded with the session.
    import glob
    runs = glob.glob(RUN + "/training/GPT_XTTS_si_female-*")
    if runs:
        latest = max(runs, key=os.path.getmtime)
        dest = KEEP + "/training/" + os.path.basename(latest)
        os.makedirs(dest, exist_ok=True)
        for name in ("config.json", "best_model.pth", "speaker_refs.json"):
            srcf = latest + "/" + name
            if os.path.isfile(srcf):
                shutil.copy2(srcf, dest + "/" + name)
                print(f"  kept {name}  {os.path.getsize(srcf)/1e9:.1f} GB")
        print("resume next session with --continue-path", dest)

    if out_of_disk:
        print("\n" + "!" * 70)
        print("training stopped early on disk pressure:", out_of_disk)
        print("The checkpoint is intact, so evaluation and export below still run")
        print("on it. To get further next session, resume from", KEEP)
        print("and raise --save-step so fewer periodic checkpoints accumulate.")
        print("!" * 70, flush=True)
    if died_of_nan:
        raise RuntimeError(
            "training diverged to nan. fp16 is off by default now; if you enabled "
            "--mixed-precision, remove it. Otherwise check the corpus for silent "
            "or corrupt clips.")


In [ ]:
# Re-run to follow along. loss_mel_ce is the acoustic reconstruction term and the
# only one that tracks audio quality; loss_text_ce carries weight 0.01.
!grep -E "loss_mel_ce|EPOCH|EVAL|BEST" /kaggle/working/train.log | tail -n 25

### Training curves

`loss_mel_ce` is the acoustic reconstruction term and the only loss that tracks audio
quality &mdash; `loss_text_ce` carries weight 0.01. Two curves, answering different
questions: **train** says whether the model is fitting at all, **eval** says whether it
is generalising. Once eval turns up while train keeps falling, every later checkpoint is
worse than one already on disk, and the export should come from `best_model.pth` rather
than the last step. The verdict printed below states which case this run is in.

In [ ]:
# Curves survive in the notebook output; the TensorBoard events do not -- they
# live in the run directory, which Kaggle deletes with the session.
sh(["python", "plot_training.py", "--log", LOG,
    "--out", "/kaggle/working/curves.png",
    "--title", "XTTS-v2 Sinhala female -- Dinithi + Harini"])

from IPython.display import Image, display
display(Image("/kaggle/working/curves.png"))


### Resuming after the 12 h limit
Turn on **Persistence &rarr; Files**. Next session, re-run cells 1&ndash;4 then this instead of
the training cell above.

In [ ]:
# The mirror written by the training cell is what survives the session, so
# resume points at KEEP and trains onward into RUN on /kaggle/temp again.
# import glob, os
# prev = max(glob.glob(KEEP + "/training/GPT_XTTS_si_female-*"), key=os.path.getmtime)
# !cd {SRC} && python train_xtts_female.py --dataset {DATASET} --out {RUN} \
#     --epochs 40 --batch-size 4 --grad-accum 16 --lr 1e-5 --continue-path {prev}

## 7. Objective evaluation

MCD, log-F0 RMSE, F0 correlation, speaker similarity, duration ratio, generation failure
rate and RTF over the held-out split. Add `--utmos` for the learned MOS predictor, and
`--asr openai/whisper-large-v3` for the CER gap (slow, large download).

In [ ]:
import glob, os
runs = glob.glob(RUN + "/training/GPT_XTTS_si_female-*")
if not runs:
    raise RuntimeError("no training run directory found -- training did not produce "
                       "a checkpoint. Check /kaggle/working/train.log.")
run = max(runs, key=os.path.getmtime)
print("run:", run)

sh(["python", "evaluate_xtts.py", "--run", run, "--base", BASE,
    "--dataset", DATASET, "--out", EVAL, "--n", "40", "--utmos"])


In [ ]:
from IPython.display import Audio, Markdown, display
import glob, json

display(Markdown(open(EVAL + "/report.md", encoding="utf-8").read()))

ref = json.load(open(DATASET + "/eval_reference.json", encoding="utf-8"))
for it in ref[:4]:
    syn = EVAL + "/synth/" + it["clip_id"] + ".wav"
    if not glob.glob(syn):
        continue
    print("\n" + it["sinhala"])
    print("  speaker:", it["speaker"])
    print("  REAL recording:");  display(Audio(DATASET + "/" + it["wav"]))
    print("  SYNTHESISED:");     display(Audio(syn))

## 8. Build the MOS / SUS listening panel

The two metrics the literature actually compares on need human ears. This writes one
self-contained HTML file &mdash; download it from the output pane and send it to native
speakers; they rate in a browser and send back a CSV.

In [ ]:
sh(["python", "listening_test.py", "--run", run, "--base", BASE,
    "--dataset", DATASET, "--out", "/kaggle/working/listening_test"])
print()
for p in sorted(os.listdir("/kaggle/working/listening_test")):
    full = "/kaggle/working/listening_test/" + p
    if os.path.isfile(full):
        print(f"  {p}  {os.path.getsize(full)/1e6:.1f} MB")


## 9. Export the model

`model.pth` + `config.json` + `vocab.json` in one folder. Always run text through
`sinhala_text.to_ascii()` before synthesising &mdash; raw Sinhala gives `[UNK]` and noise.

In [ ]:
import shutil, os, glob
EXP = "/kaggle/working/xtts_si_female"
os.makedirs(EXP, exist_ok=True)
ck = run + "/best_model.pth"
if not os.path.isfile(ck):
    ck = KEEP + "/training/" + os.path.basename(run) + "/best_model.pth"
if not os.path.isfile(ck):
    ck = max(glob.glob(run + "/checkpoint_*.pth"),
             key=lambda p: int(p.split("_")[-1].split(".")[0]))
# Hardlink where possible: on Kaggle the resume mirror and the export are both
# on /kaggle/working, and a copy would spend another 5.5 GB of the 20 GB quota
# on bytes that already exist.
try:
    os.link(ck, EXP + "/model.pth")
except (OSError, AttributeError):
    shutil.copy2(ck, EXP + "/model.pth")
for f in ("config.json", "vocab.json"):
    shutil.copy2(BASE + "/" + f, EXP + "/" + f)
shutil.copy2(CODE + "/xtts_sinhala/sinhala_text.py", EXP + "/sinhala_text.py")
print("exported from", ck)
!du -sh {EXP} && ls -la {EXP}

## 10. The next-run brief

Kaggle deletes the session shortly after it ends, so everything needed to plan the next
run is assembled now, while the files still exist. Each recommendation is **derived from
this run's numbers** and carries the evidence that produced it &mdash; it is not a
checklist. Download `next_run.md` from the output pane; it is self-contained.

In [ ]:
# Reads prepare_report.json, train.log and eval_out/metrics.json, and writes one
# markdown file. Missing inputs are reported as missing rather than raising: a
# partial brief still beats reconstructing this by hand after the session is gone.
sh(["python", "next_run_report.py",
    "--dataset", DATASET, "--log", LOG, "--eval", EVAL, "--run", run,
    "--out", "/kaggle/working/next_run.md"])

from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/next_run.md", encoding="utf-8").read()))
